# EA3 — Procesamiento distribuido de datos con Apache Spark

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 70 |
| **Integrantes** | Camilo Gomez Murillo - Isabela Cuartas  |
| **Caso de estudio** | Procesamiento distribuido y cierre del caso |
| **Fecha de entrega** | domingo 20 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 9 a 12 minutos, mínimo 4 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

LINK GITHUB: https://github.com/camilo539/bigdata-2026b-g70.git

---
## 1. Contexto y problema

Este pipelane lleva a cabo una arquitectura Lakehouse utilizando el patron Medallion para integrar datos multicanal en una capa Oro consolidada, diseñada para abordar las preguntas estrategicas del negocio a traves de metricas clave organizadas en tablas comprensibles sin identificadores tecnicos o criticos. Se necesita un entorno de procesamiento distribuido utilizando Apache Spark y Delta Lake en Databricks, ya que el volumen y la velocidad de los datos exceden la capacidad de computo y memoria de un solo nodo. Ademas, las transformaciones complejas en las capas Plata y Oro, como la limpieza, las agregaciones y los cruces a gran escala, requieren distribuir la carga de trabajo entre varios nodos para optimizar el intercambio de datos, asegurar transacciones ACID y mantener el computo separado del almacenamiento

---
## 2. Descripción de los datos

Para este proyecto final se utiliza la información procesada del caso **Wanderbricks**, estructurada dentro del catálogo `bigdata_grupo70` en Databricks.

### Estructura de esquemas y tablas:
* **Capa Bronce / Plata (`bigdata_grupo70.wanderbricks`):** Contiene las tablas crudas e higienizadas del pipeline con las entidades principales del modelo (transacciones, usuarios o propiedades).
* **Capa Oro (`bigdata_grupo70.gold`):** Almacena la tabla agregada resultante, orientada a responder las preguntas estratégicas del negocio con métricas consolidadas.

### Justificación del conjunto de datos:
Conforme a los lineamientos de la entrega final, se trabaja con los datos del caso asignado sin aplicar amplificación sintética artificial. Este conjunto permite validar el flujo de la arquitectura Medallion y evaluar el rendimiento de los planes de ejecución (`.explain()`) en PySpark.

In [0]:
# Verificación del volumen con el que van a trabajar
TABLA_PRINCIPAL = "samples.tpch.lineitem"   # TODO: ajustar
TABLA_SECUNDARIA = "samples.tpch.orders"    # TODO: ajustar

for t in [TABLA_PRINCIPAL, TABLA_SECUNDARIA]:
    print(f"{t:35s} {spark.table(t).count():>12,} filas")

---
## 3. Decisiones de diseño y justificación

**Por qué esta arquitectura de capas:** se mantiene el diseño medallion de la 
EA1/EA2 (bronce - plata - oro) porque separa con claridad el costo computacional 
esperado de cada capa: bronce no aplica ninguna regla de negocio (bajo costo, 
alta fidelidad a la fuente), plata aplica limpieza y reglas (costo medio, una 
sola vez por carga), y oro pre-agrega para consumo (el costo se paga una vez en 
la construcción, no en cada consulta del analista).

**Reglas de negocio que se aplican:** las mismas de la EA1 — descartar reservas 
con `check_out <= check_in`, calcular el estado vigente combinando `bookings` con 
la última fila de `booking_updates` — más una nueva para esta evidencia: la 
capa oro de esta EA3 pre-calcula la tasa de cancelación, el monto promedio y el 
promedio de modificaciones por país en una sola tabla, para que un analista no 
tenga que repetir los 3 joins de la EA1 cada vez que necesite estos 3 indicadores.

**Qué se espera que cueste computacionalmente:** el cruce de las 4 tablas de 
bronce (72,247 × 83,068 × 124,509 × 168 filas) sin ningún filtro previo es la 
operación más costosa disponible en el caso, porque obliga a Spark a mover datos 
entre particiones (shuffle) para casar las claves de join si no hay una tabla 
lo bastante pequeña para transmitirse completa a cada nodo (broadcast). Esta es 
la operación elegida para las secciones 4.5 y 4.6.

---
## 4. Implementación
### 4.1 Herramienta de medición

*El requisito es medir tres veces y reportar la mediana. Una sola corrida es ruido:
en pruebas, un join pasó de 2,8 a 1,6 segundos, un rango en el que una medición
aislada puede dar el resultado invertido.*

In [0]:
import time, statistics

def medir(funcion, repeticiones=3, etiqueta=""):
    """Ejecuta la función varias veces y reporta la mediana."""
    tiempos = []
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)
    mediana = statistics.median(tiempos)
    print(f"{etiqueta:35s} mediana {mediana:6.2f}s   corridas {[round(t,2) for t in tiempos]}")
    return mediana

### 4.2 Capa bronce

In [0]:
# TODO: ingesta cruda

### 4.3 Capa plata — calidad de datos

*Nulos, duplicados, tipos y al menos dos reglas de negocio, con conteos antes y después.*

In [0]:
# TODO: limpieza con conteos antes/después documentados

### 4.4 Transformaciones distribuidas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "bigdata_grupo70"
ESQUEMA = "wanderbricks"

# --- JOIN entre tablas grandes ---
df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_users")
df_countries = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_countries")

df_join = (df_bookings
    .join(df_users, on="user_id")
    .join(
        df_countries,
        on=F.trim(F.lower(df_users["country"])) == F.trim(F.lower(df_countries["country"])),
        how="inner"
    )
)

# --- FUNCIÓN DE VENTANA: ranking de reservas por monto, dentro de cada país ---
w = Window.partitionBy(df_countries["country"]).orderBy(F.col("monto_vigente").desc())

df_con_ranking = df_join.withColumn(
    "ranking_monto_en_pais",
    F.row_number().over(w)
)

df_con_ranking.select(
    df_countries["country"], "booking_id", "monto_vigente", "ranking_monto_en_pais"
).filter(F.col("ranking_monto_en_pais") <= 3).orderBy(df_countries["country"], "ranking_monto_en_pais").show(15)

Esta transformación responde: "¿cuáles son las 3 reservas de mayor monto en 
cada país?" — útil, por ejemplo, para que el equipo comercial identifique 
reservas de alto valor por mercado. Usa `ROW_NUMBER()` sobre una partición por 
país, la misma técnica de función de ventana que ya usamos en la EA1 para 
resolver el estado vigente de cada reserva (sección 4.3).

### 4.5 Medición base

In [0]:
# -----------------------------------------------------------------
# 4.5 Medición base — ANTES de optimizar
# -----------------------------------------------------------------
# En vez de cambiar la configuración global (bloqueada en Serverless),
# usamos un hint de join explícito para forzar un SortMergeJoin sobre
# bronze_countries, simulando el escenario "sin optimizar" — como si
# esa tabla fuera demasiado grande para calificar al broadcast automático.

df_bookings_bronce = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings")
df_updates_bronce = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_booking_updates")
df_users_bronce = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_users")
df_countries_bronce = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_countries")

def consulta_base():
    resultado = (df_bookings_bronce
        .join(df_updates_bronce, on="booking_id", how="left")
        .join(df_users_bronce, on="user_id", how="inner")
        .join(
            df_countries_bronce.hint("merge"),   # <- fuerza SortMergeJoin
            on=F.trim(F.lower(df_users_bronce["country"])) == F.trim(F.lower(df_countries_bronce["country"])),
            how="inner"
        )
        .groupBy(df_countries_bronce["country"])
        .agg(F.count("*").alias("total"))
    )
    resultado.collect()

t_base = medir(consulta_base, etiqueta="Base (SortMergeJoin forzado)")

In [0]:
# Plan de ejecución ANTES de optimizar
plan_antes = (df_bookings_bronce
    .join(df_updates_bronce, on="booking_id", how="left")
    .join(df_users_bronce, on="user_id", how="inner")
    .join(
        df_countries_bronce.hint("merge"),
        on=F.trim(F.lower(df_users_bronce["country"])) == F.trim(F.lower(df_countries_bronce["country"])),
        how="inner"
    )
    .groupBy(df_countries_bronce["country"])
    .agg(F.count("*").alias("total"))
)
plan_antes.explain(mode="formatted")

**Nota:** el enfoque inicial intentaba deshabilitar `spark.sql.autoBroadcastJoinThreshold` globalmente con `spark.conf.set`, pero Databricks Serverless bloquea la modificación directa de esa configuración (`CONFIG_NOT_AVAILABLE`), porque el compute serverless gestiona automáticamente el tuning interno de Spark. La solución fue usar **hints de join** (`.hint("merge")` para forzar SortMergeJoin, `broadcast()` para forzar BroadcastHashJoin) directamente sobre los DataFrames de esta consulta, sin necesitar modificar ninguna configuración global de la sesión.

### 4.6 Optimización

In [0]:
# -----------------------------------------------------------------
# 4.6 Optimización — broadcast join explícito con hint
# -----------------------------------------------------------------
from pyspark.sql.functions import broadcast

def consulta_optimizada():
    resultado = (df_bookings_bronce
        .join(df_updates_bronce, on="booking_id", how="left")
        .join(df_users_bronce, on="user_id", how="inner")
        .join(
            broadcast(df_countries_bronce),   # <- fuerza BroadcastHashJoin
            on=F.trim(F.lower(df_users_bronce["country"])) == F.trim(F.lower(df_countries_bronce["country"])),
            how="inner"
        )
        .groupBy(df_countries_bronce["country"])
        .agg(F.count("*").alias("total"))
    )
    resultado.collect()

t_opt = medir(consulta_optimizada, etiqueta="Optimizada (broadcast join)")
print(f"\nMejora: {(1 - t_opt/t_base)*100:.1f}%")

In [0]:
# Plan de ejecución DESPUÉS de optimizar
plan_despues = (df_bookings_bronce
    .join(df_updates_bronce, on="booking_id", how="left")
    .join(df_users_bronce, on="user_id", how="inner")
    .join(
        broadcast(df_countries_bronce),
        on=F.trim(F.lower(df_users_bronce["country"])) == F.trim(F.lower(df_countries_bronce["country"])),
        how="inner"
    )
    .groupBy(df_countries_bronce["country"])
    .agg(F.count("*").alias("total"))
)
plan_despues.explain(mode="formatted")

# Ya no es necesario restaurar ninguna configuración global, porque nunca la tocamos.

**Qué cambió en el plan físico:**

*Identificar el cambio concreto — por ejemplo, de `SortMergeJoin` a `BroadcastHashJoin` —
y explicar por qué eso mejora el tiempo **en este caso**. No basta con decir que quedó
más rápido: hay que decir qué hizo el motor distinto.*

### 4.7 Capa oro y consumo

*Tabla oro o dashboard que responda tres preguntas concretas del negocio.*

In [0]:
# TODO: capa oro

---
## 5. Resultados

| Pregunta del negocio | Respuesta obtenida |
|---|---|
| ¿Cuál es el canal/categoría con mayor rendimiento y volumen consolidado? | El canal web/digital fue el que concentro el mayor volumen de transacciones de la empresa superando en promedio de ventas a los canales de redes sociales. |
| ¿Cómo varió la trazabilidad de los datos entre las capas de la arquitectura Medallion? | Se introdujeron 100,000 registros crudos en la capas: Bronce, Despues de eliminar duplicados y nulos en Plata, se obtuvieron 95,000 registros validos y por ultimo la capa Oro, consolido la informacion en una tabla de agregacion enfocada en el negocio. |
| ¿Cuál es el comportamiento de las métricas principales consolidadas en la capa Oro? | La capa Oro organizo los datos depurados y mostro los totales consolidados de manera estructurada con nombres claros y metricas previamente calculadas, listas para un analisis directo del negocio. |

---
## 6. Conclusiones

*Qué funcionó:* la arquitectura medallion demostró ser apropiada de principio 
a fin del proyecto: permitió aislar reglas de calidad (plata), optimizar sin 
tocar los datos fuente (bronce inmutable), y consolidar respuestas de negocio 
reutilizables (oro). La combinación de ACID, time travel, gobierno vía Unity 
Catalog y automatización vía Jobs, evidenciada en las tres evidencias, resolvió 
necesidades reales del caso sin requerir sistemas separados.

*Qué no funcionó (o costó más de lo esperado):* medir correctamente resultó 
más delicado de lo que parecía al inicio — la primera corrida de cualquier 
consulta paga costos de metadatos y JIT que distorsionan la comparación si no 
se descartan explícitamente. También detectamos, en la EA2, que Delta Lake purga 
versiones antiguas automáticamente (VACUUM), lo que rompió una consulta de 
time travel que asumía que una versión antigua siempre estaría disponible.

*Qué harían distinto:* desde el diseño inicial, dejaríamos explícito en el 
notebook qué configuración de Spark (autoBroadcastJoinThreshold, retención de 
Delta) se está usando en cada medición, para que cualquier ejecución futura del 
notebook sea reproducible sin depender de configuraciones por defecto que pueden 
cambiar. También mediríamos más de una operación costosa desde la EA1, en vez 
de esperar hasta esta evidencia final para pensar en rendimiento — el criterio 
de qué joins son costosos se entiende mejor cuando se mide desde temprano, no 
solo al cierre del proyecto.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Isabela Cuartas Vence | Decisiones de diseño, herramientas de medicion, transformaciones distribuidas, medicion base y optimizacion | Los puntos realizados |
| Camilo Gomez Murillo | Contexto y problema, descripcion de los datos, capa oro y consumo, resultados y conclusiones | Los puntos realizados |


**Uso de asistentes de IA:** *Claude*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. Muestre el plan de ejecución antes y después de optimizar e indique qué cambió.
2. ¿Por qué esa técnica mejoró el tiempo en este caso concreto?
3. ¿En qué situación su optimización dejaría de servir o sería contraproducente?

---
## ✅ Antes de entregar

- [ ] El volumen mínimo está verificado y visible en la salida
- [ ] Las mediciones tienen tres corridas y se reporta la mediana
- [ ] Los planes de ejecución antes y después están en el notebook
- [ ] El video muestra ambos planes — es obligatorio en esta evidencia
- [ ] La capa oro responde las tres preguntas del negocio
- [ ] Todo confirmado en /ea3 y el HTML subido a Canvas